# Amber Terminal from Python

Bloomberg's notebooks (BQuant) sit on top of its data. This is the same idea on Amber Terminal's free data: a few lines of pandas against the API on your own computer.

**Before you start**

1. Start Amber Terminal: `node scripts/launch.js` in the amber-terminal folder. Keep it running.
2. Check the address it prints (usually `http://localhost:3787`) and put it in the first cell below.
3. You need pandas, and matplotlib for the chart: `pip install pandas matplotlib`.

Every number comes from a public source with no key. Some prices run up to 15 minutes behind.

In [ ]:
import json
import urllib.request
from urllib.parse import quote

import pandas as pd

BASE = "http://localhost:3787"  # the address Amber printed when it started


def get(path):
    """Any of Amber's JSON endpoints, as a Python dict."""
    with urllib.request.urlopen(BASE + path) as r:
        return json.load(r)


about = get("/api/about")
print(f"Connected to Amber Terminal {about['version']} in {about['dir']}")

## 1. History: BDH

`/api/bdh` is Bloomberg's `BDH` as a web address: one security, the fields you want, a start and an end, and `per` for daily (D), weekly (W), monthly (M), quarterly (Q) or yearly (Y) rows. Dates can be `2025-01-31` or relative, like `-1Y` for a year ago.

Securities are written the terminal's way: `AAPL US EQUITY`, `SPX INDEX`, `GC1 COMDTY`, `EURUSD CURNCY`, `XBTUSD CURNCY`.

Here's a year of daily closes for four of them, side by side.

In [ ]:
def bdh(security, fields="PX_LAST", start="-1Y", end="", per="D"):
    url = (f"{BASE}/api/bdh?security={quote(security)}&fields={fields}"
           f"&start={start}&end={end}&per={per}&format=csv")
    return pd.read_csv(url, parse_dates=["date"], index_col="date")


securities = ["SPX INDEX", "AAPL US EQUITY", "GC1 COMDTY", "XBTUSD CURNCY"]
closes = pd.concat({s: bdh(s)["PX_LAST"] for s in securities}, axis=1)
closes.tail()

Weekly bars with every price field, the way you'd pull them into a model:

In [ ]:
bdh("AAPL US EQUITY", fields="PX_OPEN,PX_HIGH,PX_LOW,PX_LAST,PX_VOLUME", start="-3M", per="W").tail()

## 2. A snapshot: BDP

`/api/bdp` is Bloomberg's `BDP`: one row per security, one column per field. Market cap, sector, dividend yield and P/E are filled for US-listed stocks; the price fields work for everything. A field it can't fill comes back empty.

In [ ]:
def bdp(securities, fields):
    url = (f"{BASE}/api/bdp?securities={quote(','.join(securities), safe=',')}"
           f"&fields={','.join(fields)}&format=csv")
    return pd.read_csv(url, index_col="security")


bdp(["AAPL US EQUITY", "MSFT US EQUITY", "NVDA US EQUITY", "SPX INDEX"],
    ["NAME", "PX_LAST", "CHG_PCT_1D", "CHG_PCT_YTD", "CUR_MKT_CAP", "PE_RATIO"])

## 3. One chart

The four closes from step 1, each rebased to 100 on the first day they all traded, so you can compare them on one scale. Bitcoin trades at weekends and the others don't, so only the shared days are kept.

In [ ]:
shared = closes.dropna()
rebased = shared / shared.iloc[0] * 100
rebased.plot(title="The last year, rebased to 100", figsize=(10, 4), grid=True);

## 4. Any screen's data: WIRP and GC

Every function in the terminal has its data at `/api/fn/<code>`. WIRP is the Fed rate path the futures market is pricing, one row per FOMC meeting: the rate before and after, the step in basis points and the total move so far.

In [ ]:
wirp = get("/api/fn/wirp")
fed = pd.DataFrame(wirp["meetings"])[["date", "before", "after", "stepBp", "cumBp"]]
print(wirp["source"])
fed

GC is the US Treasury yield curve, today against a week, a month and a year ago, plus the 2s10s spread (the 10-year yield minus the 2-year, in basis points).

In [ ]:
gc = get("/api/fn/gc")
curve = pd.DataFrame({c["label"]: c["values"] for c in gc["curves"]}).loc[gc["tenors"]]
print(f"2s10s spread: {gc['spread']['bp']} bp")
curve

## Next

- The other codes work the same way: `/api/fn/btmm`, `/api/fn/ecst`, `/api/fn/des?symbol=AAPL`, `/api/fn/omon?symbol=AAPL`. Type `API` in the terminal for live addresses.
- Rather ask in plain English? Type `ASK` in the terminal to connect your own Claude.